# Train the motion ball detector on a Kaggle GPU

The new ball detector reads **motion**, not colours: each input is a 320 x 320 crop at
full resolution with three channels, the grey image, the difference from the
background at frame *t* and the same at *t - 1*
(`tracking/ball_online.py`). This notebook builds its dataset from the clips, trains
a small YOLO on it, exports it for the real-time pipeline and measures it on the T4.

A saved Kaggle version runs in the background: once it has started you can close the
browser and turn the computer off.

**Setup**
1. On your PC, after the offline pipeline has run on the clips:
   `python training/pack_kaggle_inputs.py` writes `kaggle_inputs.zip` (the ball,
   player and court CSVs of every clip, plus `ground_truth/ball_*_gt.csv`, your
   manual labels included).
2. *Input -> Add Input -> Upload*: a **private** dataset with the videos (`data/*.mp4`,
   names unchanged) and `kaggle_inputs.zip` (Kaggle unpacks it). The dataset of the
   detection notebook can simply get a new version with the zip added.
3. *Settings*: *Accelerator* GPU (T4), *Internet* on. Then *Save Version -> Save & Run All*.

## 1. Check the GPU
If `CUDA available: False`, set *Settings -> Accelerator* to a GPU and run again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Get the code
Clones the branch with the motion detector. albumentations is removed: ultralytics
would apply its grey / CLAHE transforms, which mix the channels.

In [ ]:
BRANCH = 'realtime-motion-ball'
!git clone -q -b {BRANCH} https://github.com/marcomassa-unitn/tennis_players_tracking.git
%cd tennis_players_tracking
!pip -q install ultralytics onnx onnxslim onnxruntime-gpu
!pip -q uninstall -y albumentations

## 3. Find the videos and the CSVs
They are searched for in all the inputs, so the name of the dataset does not matter.
The CSVs go back to their place in the project (`outputs/<clip>/...`,
`ground_truth/`).

In [ ]:
import glob, os, shutil
files = glob.glob('/kaggle/input/**/*', recursive=True)
VIDEOS = sorted(f for f in files if f.endswith('.mp4'))
VIDEO_DIRS = sorted({os.path.dirname(v) for v in VIDEOS})
assert VIDEOS, 'No .mp4 video found: add the videos to the dataset attached to this notebook'

copied = 0
for f in files:
    f2 = f.replace(os.sep, '/')
    for root in ('outputs/', 'ground_truth/'):
        if '/' + root in f2 and f2.endswith('.csv'):
            dst = root + f2.split('/' + root, 1)[1]
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(f, dst)
            copied += 1
assert copied, 'No CSV found: add kaggle_inputs.zip to the dataset (training/pack_kaggle_inputs.py)'
print(len(VIDEOS), 'videos in', VIDEO_DIRS)
print(copied, 'CSVs copied;', 'clips with a ball track:',
      sorted(os.path.basename(os.path.dirname(os.path.dirname(p)))
             for p in glob.glob('outputs/*/ball_coordinates/ball_*.csv') if 'candidates' not in p))

## 4. Build the dataset
Each clip is replayed with the online background of the live pipeline; positives
come from the reliable detections of the current tracker and from the manual labels,
negatives from frames where the ball is known to be elsewhere. The test clips
(Input_video2, input_video, input11) are never written. A few minutes.

In [ ]:
!python training/build_ball_dataset.py --videos {','.join(VIDEO_DIRS)} --preview

In [ ]:
# A look at the crops: the ball at t in green, at t-1 in red, the grey image in blue.
from IPython.display import Image, display
for p in sorted(glob.glob('datasets/ball_motion/preview_*.jpg'))[:3]:
    print(p); display(Image(p, width=900))

## 5. Train
YOLO11n at 320 px, about 15 minutes on a T4. Use `--model yolov8s.pt` for a larger
model if the recall is not enough.

To skip the training and only export and measure weights you already have, add
them to the dataset as `ball_motion.pt`. With `ball_tracker.pt` (the offline
YOLOv8x) in the dataset too, the pipeline is also measured with the colour
fallback by the players.

In [ ]:
GIVEN = [f for f in files if os.path.basename(f) == 'ball_motion.pt']
if GIVEN:
    BEST = GIVEN[0]
    print('Using the given weights, no training:', BEST)
else:
    !python training/train_ball_motion.py --data datasets/ball_motion/data.yaml --model yolo11n.pt --device 0
    found = sorted(glob.glob('runs/**/weights/best.pt', recursive=True), key=os.path.getmtime)
    assert found, 'Training produced no best.pt: see the log above'
    BEST = found[-1]
    print('Trained weights:', BEST)
    # ultralytics strips the optimizer from the weights only when training ends
    # normally: if it is still there, the run stopped early and BEST is partial.
    import pandas as pd
    ckpt = torch.load(BEST, map_location='cpu', weights_only=False)
    epochs = len(pd.read_csv(os.path.join(os.path.dirname(os.path.dirname(BEST)), 'results.csv')))
    if ckpt.get('optimizer') is not None:
        print(f'!!! TRAINING STOPPED EARLY after {epochs} epochs: these are the best weights so far.')
        print('!!! Look in the log above for the error (out of memory, time limit...).')
    else:
        print(f'Training finished normally after {epochs} epochs.')

## 6. Export
ONNX (for onnxruntime on any machine, CPU or CUDA) and TensorRT FP16 (only for this
GPU model, built here).

In [ ]:
os.makedirs('models', exist_ok=True)
shutil.copy(BEST, 'models/ball_motion.pt')       # a writable copy (inputs are read-only)
BEST = 'models/ball_motion.pt'
!python training/export_ball_model.py --weights {BEST} --format onnx
ONNX = 'models/ball_motion.onnx'
assert os.path.exists(ONNX), 'ONNX export failed: see the log above'
# TensorRT is optional: if its export fails, the rest goes on with ONNX only.
!python training/export_ball_model.py --weights {BEST} --format engine
ENGINE = 'models/ball_motion.engine'
FALLBACKS = [f for f in files if os.path.basename(f) == 'ball_tracker.pt']
FALLBACK = FALLBACKS[0] if FALLBACKS else None
# PyTorch first: it always gets the GPU, while onnxruntime-gpu may not match the
# CUDA of the image (then it silently runs on the CPU: see the check below).
MODELS = [('torch-cuda', BEST, None)]
if FALLBACK:          # the colour detector by the players (see README, Real time)
    MODELS.append(('torch-cuda-fallback', BEST, FALLBACK))
MODELS.append(('onnx', ONNX, None))
if os.path.exists(ENGINE):
    MODELS.append(('tensorrt', ENGINE, None))
    MODELS.append(('tensorrt-search2', ENGINE, None))   # lost ball searched every 2nd frame
print('Runs to measure:', [m[0] for m in MODELS])

## 7. Run the real-time pipeline on the test clips
The whole analysis, one causal pass, every frame (`--clock all`): `summary.json` says
how many frames per second it sustains after the startup, stage by stage. Then the
ball against the ground truth.

In [ ]:
TEST = [v for v in VIDEOS if os.path.splitext(os.path.basename(v))[0] in ('Input_video2', 'input_video', 'input11')]
import json
import onnxruntime as ort
# A GPU that fails here (e.g. 'CUDA failure 999') makes onnxruntime fall back to the
# CPU: the fps below would then be CPU fps. Say it loudly.
sess = ort.InferenceSession(ONNX, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
ON_GPU = 'CUDAExecutionProvider' in sess.get_providers()
print('onnxruntime providers:', sess.get_providers())
if not ON_GPU:
    print('!!! onnxruntime is NOT using the GPU: the timings below are CPU timings.')
    print('!!! Check nvidia-smi; a new version usually gets a working machine.')
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used --format=csv
for name, model, fallback in MODELS:
    extra = f'--fallback "{fallback}"' if fallback else ''
    search_every = 2 if name.endswith('search2') else 1
    for v in TEST:
        stem = os.path.splitext(os.path.basename(v))[0]
        !python realtime_pipeline.py --video "{v}" --model "{model}" {extra} --calibrate --search tiles --search-every {search_every} --output outputs/realtime_{name} --log-every 0
        path = f'outputs/realtime_{name}/{stem}/realtime/summary.json'
        if not os.path.exists(path):
            print(f'{name} {stem}: FAILED, see the log above')
            continue
        s = json.load(open(path))
        print(f">>> {name:10s} {stem:13s} {s['steady_fps']:6.1f} fps  real time: {s['real_time']}  "
              + '  '.join(f"{k} {st['mean_ms']:.1f}ms" for k, st in s['stages'].items() if k != 'court_fit'))

In [ ]:
!python evaluation/benchmark_ball.py --videos {",".join(VIDEO_DIRS)} --realtime outputs/realtime_torch-cuda --model {BEST} --csv outputs/benchmark_ball.csv
if FALLBACK:
    !python evaluation/benchmark_ball.py --videos {",".join(VIDEO_DIRS)} --realtime outputs/realtime_torch-cuda-fallback --csv outputs/benchmark_ball_fallback.csv

## 8. Save the results
Only `/kaggle/working` is kept. Download from the *Output* section of the saved
version and put `best.onnx` in `models/ball_motion.onnx` on your PC (the `.engine`
only works on a T4).

In [ ]:
out = '/kaggle/working'
results = sorted(glob.glob('runs/**/results.csv', recursive=True), key=os.path.getmtime)[-1:]
for f in [BEST, ONNX, ENGINE, *results, 'outputs/benchmark_ball.csv', 'outputs/benchmark_ball_fallback.csv',
          *glob.glob('datasets/ball_motion/preview_*.jpg')]:
    if os.path.exists(f):
        print('Saved', shutil.copy(f, out))
for f in glob.glob('outputs/realtime_*/*/realtime/summary.json'):
    run, clip = f.split('/')[1], f.split('/')[2]
    print('Saved', shutil.copy(f, f'{out}/summary_{run}_{clip}.json'))
%cd /kaggle/working
!rm -rf tennis_players_tracking